# Tensor Math Filling Demo (Candle)

This notebook showcases generating tensors from analytical mathematical expressions using Candle.
We build coordinate grids and evaluate formulas like: 
- $f(x,y) = in(a x) os(b y)$
- $g(x,y) = e^{-((x-x_0)^2 + (y-y_0)^2)/(2\sigma^2)}$ (Gaussian bump)
- Radial distance: $r(x,y) = qrt{(x-x_c)^2 + (y-y_c)^2}$ (normalized)
- Checkerboard: $c(x,y) = ((floor x / s 
floor + floor y / s 
floor) \bmod 2)$

We'll also parse free-form expressions (future: via `arithmetic_parser`) to turn strings into tensor values.

In [2]:
// Imports
use candle::{Result, Tensor, DType, Device};
use std::f64::consts::PI;
let device = Device::Cpu;
println!("Device: {:?}", device);

Error: unresolved import `candle`

Error: unresolved import `candle`

Error: unresolved import `candle`

Error: unresolved import `candle`

## Coordinate Grid Helper
We create a function to build 2D coordinate grids X, Y over ranges with optional centering.

In [ ]:
fn make_xy(nx: usize, ny: usize, x_min: f64, x_max: f64, y_min: f64, y_max: f64, device: &Device) -> Result<(Tensor, Tensor)> {
    let xs: Vec<f64> = (0..nx).map(|i| x_min + (x_max - x_min) * (i as f64) / (nx as f64 - 1.0)).collect();
    let ys: Vec<f64> = (0..ny).map(|j| y_min + (y_max - y_min) * (j as f64) / (ny as f64 - 1.0)).collect();
    let x = Tensor::new(&xs, device)?.reshape((nx, 1))?;
    let y = Tensor::new(&ys, device)?.reshape((1, ny))?;
    // broadcast to (nx, ny)
    let X = x.broadcast_as((nx, ny))?;
    let Y = y.broadcast_as((nx, ny))?;
    Ok((X, Y))
}
let (nx, ny) = (128, 128);
let (X, Y) = make_xy(nx, ny, -1.0, 1.0, -1.0, 1.0, &device)?;
println!("Grid shape: {:?}", X.shape());

## Analytical Functions
We'll build sample tensors using direct Rust math with Candle broadcasting.

In [ ]:
// sin-cos pattern f(x,y) = sin(a x) * cos(b y)
let a = 6.0f64;
let b = 4.0f64;
let f = (X.clone() * a)? .sin()? * (Y.clone() * b)?.cos()?;
println!("f stats -> min: {:.3} max: {:.3}", f.min_reduce()?.to_scalar::<f64>()?, f.max_reduce()?.to_scalar::<f64>()?);
// Gaussian bump centered at (0,0) with sigma=0.35
let sigma = 0.35f64;
let r2 = (X.clone().powf(2.0)? + Y.clone().powf(2.0)?)?;
let g = (-(&r2 / (2.0 * sigma * sigma))?)?.exp()?;
println!("g stats -> min: {:.3} max: {:.3}", g.min_reduce()?.to_scalar::<f64>()?, g.max_reduce()?.to_scalar::<f64>()?);
// Radial distance normalized
let r = r2.sqrt()?;
let r_norm = (&r / r.max_reduce()?)?;
// Checkerboard with scale s
let s = 0.15;
let cb = ((&X / s)?.floor()? + (&Y / s)?.floor()?)? % 2.0?;
println!("checkerboard unique approx: {:?}", cb.flatten_all()?.to_vec1::<f64>()?.iter().copied().fold(std::collections::BTreeSet::new(), |mut acc, v| { let vv = if v.abs() < 1e-9 {0.0} else {1.0}; acc.insert(vv); acc }).len());

## Normalization Helper for Image Saving
We map a tensor to 0..255 u8 for quick visualization (future: integrate with image store).

In [ ]:
fn to_u8(t: &Tensor) -> Result<Vec<u8>> {
    let min = t.min_reduce()?;
    let max = t.max_reduce()?;
    let span = (&max - &min)?;
    // Avoid div-by-zero: if span==0 just zeros
    let norm = if span.to_scalar::<f64>()? == 0.0 {
        (t - &min)?
    } else {
        (t - &min)? / span?
    };
    let scaled = (norm * 255.0)?;
    Ok(scaled.to_vec1::<f64>()?.into_iter().map(|v| v.round().clamp(0.0,255.0) as u8).collect())
}
let f_u8 = to_u8(&f)?;
println!("u8 buffer length (f): {}", f_u8.len());

## (Planned) Expression Parsing
Future step: integrate `arithmetic_parser` so a user string like `sin(6*x)*cos(4*y)` is parsed and evaluated over X,Y.
We would:
1. Create a context binding symbols x,y,pi.
2. Parse expression to an AST.
3. Evaluate per element (or vectorize by mapping over flattened arrays, then reshaping).
Once stable, wrap in a helper `eval_expr(expr: &str, bindings: &HashMap<&str,&Tensor>)`.

## Next Ideas
- Integrate image saving to `images_store/`.
- Add dynamic parameter sliders (in egui context) for a, b, sigma.
- Add 3D stack: vary t and evaluate f_t(x,y) = sin(a x + t)*cos(b y - t).
- Inline LaTeX rendering for formulas to keep code + math in sync.

## Parsed Expression Integration (B3)

Below we demonstrate the new expression parser (`ExprEnv`, `eval_expr`) on the same grids used earlier.

Goals:
- Define grids X, Y (already created earlier) and parameter map.
- Evaluate a list of expression strings.
- Compare each parsed result to a manual construction when applicable.
- Show max absolute difference.
- Gaussian example using parameters (a, b, sigma).
- Timing snapshot.

We'll re-create X, Y if not in scope (idempotent).

In [ ]:
// Recreate grid (idempotent) and import parser
use std::collections::HashMap;
use candle::{Tensor, Device, DType};
use candle_notebooks::expr::{ExprEnv, eval_expr};

let dev = Device::Cpu;
let n = 128i64;
let xs: Vec<f32> = (0..n).map(|i| (i as f32)/(n as f32 - 1.0) * 2.0 - 1.0).collect();
let ys = xs.clone();
let x = Tensor::from_vec(xs.clone(), (n,), &dev)?.to_dtype(DType::F32)?;
let y = Tensor::from_vec(ys.clone(), (n,), &dev)?.to_dtype(DType::F32)?;
let xx = x.unsqueeze(0)?.broadcast_as((n, n))?; // row-wise
let yy = y.unsqueeze(1)?.broadcast_as((n, n))?; // column-wise

let mut params = HashMap::new();
params.insert("a".to_string(), 1.0f64);
params.insert("b".to_string(), 1.0f64);
params.insert("sigma".to_string(), 0.35f64);

let env = ExprEnv::new(xx.clone().to_dtype(DType::F64)?, yy.clone().to_dtype(DType::F64)?, params)?;
println!("Grid & env ready: shape={:?}", env.x.shape());

In [ ]:
// Helper to evaluate an expression and optionally compare against a manual tensor builder closure.
use std::time::Instant;

fn eval_and_compare<F>(label: &str, expr: &str, env: &ExprEnv, manual: Option<F>) -> candle_core::Result<()>
where F: Fn() -> candle_core::Result<Tensor>
{
    let t0 = Instant::now();
    let parsed = eval_expr(expr, env)?; // returns f32 tensor
    let dt = t0.elapsed();
    if let Some(build) = manual {
        let manual_t = build()?; // assume already f32 shape match
        let diff = (&parsed - &manual_t)?.abs()?.max_all()?;
        println!("{label}: expr='{}' max_abs_diff={:.3e} time_ms={:.2}", expr, diff.to_scalar::<f32>()?, dt.as_secs_f64()*1e3);
    } else {
        println!("{label}: expr='{}' time_ms={:.2}", expr, dt.as_secs_f64()*1e3);
    }
    Ok(())
}

// Prepare some manual comparison builders (capturing xx, yy)
let xx_f32 = env.x.to_dtype(DType::F32)?; // already f64 inside env
let yy_f32 = env.y.to_dtype(DType::F32)?;

// Run a few baseline expressions

eval_and_compare("radial", "sqrt(x*x + y*y)", &env, Some(|| Ok((( &xx_f32 * &xx_f32 )? + ( &yy_f32 * &yy_f32 )?)?.sqrt()? )))?;

eval_and_compare("checker", "(sin(10*x)*sin(10*y))", &env, Some(|| Ok((xx_f32.sin()? * yy_f32.sin()?)? )))?;

eval_and_compare("gaussian_basic", "exp(-(x*x + y*y))", &env, None)?;

In [ ]:
// Gaussian with parameters a, b, sigma in params
eval_and_compare(
    "gaussian_param",
    "a * exp(-((x*x)/(2*sigma*sigma) + (y*y)/(2*sigma*sigma))) * b",
    &env,
    None,
)?;

// More complex combo expression

eval_and_compare(
    "combo",
    "sin(3*x) * cos(5*y) + 0.25 * exp(-(x*x + y*y))",
    &env,
    None,
)?;

println!("Done B3 expression evaluations.");